# 🦺 Detecção de Ferrugem em Estruturas — Segmentação por Cor em HSV

**Disciplina:** IA Aplicada à Construção Civil  
**Módulo:** Visão Computacional — Segmentação por Cor e Morfologia

---

## Contexto

A **corrosão de armaduras** é uma das patologias mais críticas em estruturas de concreto armado. Sua detecção precoce evita falhas estruturais e reduz drasticamente o custo de manutenção (relação de Sitter: cada R$ 1 de prevenção evita R$ 125 de recuperação estrutural).

Fotografias de inspeção registram a ferrugem como manchas com cor característica — laranja, marrom-avermelhado — distinguível do concreto cinza, da forma metálica e do entorno. Isso torna a **segmentação por cor** uma abordagem eficiente para triagem automatizada de imagens de obra.

### Por que HSV e não RGB?

No espaço **RGB**, a cor de um pixel depende simultaneamente da pigmentação do material *e* da iluminação. Um tijolo laranja na sombra tem valores RGB completamente diferentes do mesmo tijolo ao sol.

O espaço **HSV** separa esses fatores:

| Canal | O que representa | Varia com iluminação? |
|-------|------------------|-----------------------|
| **H** (Hue / Matiz) | A cor em si (0–179 no OpenCV) | ❌ Não |
| **S** (Saturation) | Quão "viva" é a cor | Um pouco |
| **V** (Value / Brilho) | Luminosidade do pixel | ✅ Sim |

Ao definir um intervalo em H e S e deixar V livre (ou com mínimo baixo), capturamos a ferrugem sob diferentes condições de luz sem precisar retreinar nada.

### Faixas de referência para ferrugem (OpenCV: H 0–179)

| Tom | H mín | H máx | S mín | Observação |
|-----|-------|-------|-------|------------|
| Laranja vivo | 5 | 20 | 100 | Ferrugem recente, úmida |
| Marrom-ferrugem | 5 | 30 | 80 | Ferrugem seca, mais comum em obra |
| Vermelho-óxido | 0 | 10 + 160–179 | 60 | Requer duas máscaras (H faz wrap em 0) |

> 💡 **Regra prática:** comece com `hmin=5, hmax=30, smin=80, vmin=40` e refine na **Seção 5** com a grade de parâmetros.

---

## Estrutura deste notebook

| Seção | O que você vai fazer |
|-------|----------------------|
| 1 | Testar o ambiente com imagem sintética (sem Drive) |
| 2 | Montar o Google Drive e configurar caminhos |
| 3 | Processar todas as imagens em lote |
| 4 | Visualizar grid de resultados e métricas |
| 5 | Calibrar parâmetros HSV com grade interativa |

> 💡 **Recomendação:** execute a Seção 1 primeiro — ela não depende de nenhum arquivo externo.

---
## Seção 1 — Teste com imagem sintética

Criamos uma imagem que simula uma superfície de concreto com manchas de ferrugem:

- **Fundo cinza** → concreto
- **Mancha laranja-marrom** → ferrugem ativa
- **Região escura** → sombra (para testar robustez do canal V)
- **Ruído gaussiano** → variação de textura

Se o pipeline estiver correto, a máscara deve cobrir **somente** as manchas laranja-marrom, ignorando cinza, preto e branco.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# ── Criar imagem sintética 512×512 (BGR)
h, w = 512, 512
img_sint = np.full((h, w, 3), (150, 150, 150), dtype=np.uint8)  # concreto cinza

# Mancha de ferrugem central (laranja-marrom em BGR: B=20, G=80, R=180)
cx, cy, rx, ry = 256, 200, 120, 80
for i in range(h):
    for j in range(w):
        if ((j - cx)**2 / rx**2 + (i - cy)**2 / ry**2) < 1:
            img_sint[i, j] = [20, 80, 180]  # ferrugem

# Segunda mancha menor (região escurecida — ferrugem na sombra)
cv2.ellipse(img_sint, (380, 360), (60, 45), 30, 0, 360, (10, 50, 120), -1)

# Ruído gaussiano
ruido = np.random.normal(0, 10, img_sint.shape).astype(np.int16)
img_sint = np.clip(img_sint.astype(np.int16) + ruido, 0, 255).astype(np.uint8)

# ── Pipeline de detecção
def detectar_ferrugem(img_bgr, hmin=5, hmax=30, smin=80, vmin=40):
    blur = cv2.GaussianBlur(img_bgr, (5, 5), 0)
    hsv  = cv2.cvtColor(blur, cv2.COLOR_BGR2HSV)

    # CLAHE no canal V (corrige iluminação desigual)
    h_ch, s_ch, v_ch = cv2.split(hsv)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    hsv   = cv2.merge([h_ch, s_ch, clahe.apply(v_ch)])

    # Máscara de cor
    lower = np.array([hmin, smin, vmin], dtype=np.uint8)
    upper = np.array([hmax, 255, 255],  dtype=np.uint8)
    mask  = cv2.inRange(hsv, lower, upper)

    # Limpeza morfológica
    k    = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  k, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k, iterations=2)

    # Overlay laranja nos pixels detectados
    overlay = img_bgr.copy()
    px = img_bgr[mask > 0]                                   # shape (N, 3)
    cor = np.full_like(px, [0, 140, 255], dtype=np.uint8)    # mesmo shape
    overlay[mask > 0] = cv2.addWeighted(px, 0.3, cor, 0.7, 0)

    # Contornos
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contoured   = overlay.copy()
    cv2.drawContours(contoured, contours, -1, (0, 0, 255), 2)

    rust_pct = 100.0 * int(mask.sum() / 255) / mask.size
    return mask, overlay, contoured, rust_pct

mask, overlay, contoured, pct = detectar_ferrugem(img_sint)

# ── Visualização
fig, eixos = plt.subplots(1, 4, figsize=(18, 4.5))
titulos = ['Original (sintética)', 'Espaço HSV', 'Máscara de ferrugem', f'Overlay + contornos\n{pct:.1f}% de área corroída']
imgs    = [img_sint, cv2.cvtColor(cv2.cvtColor(img_sint, cv2.COLOR_BGR2HSV), cv2.COLOR_HSV2BGR), mask, contoured]
cmaps   = [None, None, 'gray', None]

for ax, im, t, c in zip(eixos, imgs, titulos, cmaps):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB) if c is None else im, cmap=c)
    ax.set_title(t, fontsize=10)
    ax.axis('off')

plt.suptitle('✅ Teste de ambiente — imagem sintética', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'✅ Ambiente OK — área corroída detectada: {pct:.2f}%')
print('   Pode prosseguir para a Seção 2.')

---
## Seção 2 — Montar o Google Drive e configurar caminhos

A célula abaixo solicita autorização para acessar o Google Drive. Clique no link gerado, selecione sua conta Google e cole o código de autorização.

> ⚠️ **Pasta compartilhada?** Se `fotos_obra` não estiver em *Meu Drive*:  
> *Drive → clique direito na pasta → Organizar → Adicionar atalho ao Drive*  
> O caminho `/content/drive/MyDrive/fotos_obra` passará a funcionar normalmente.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive montado.')

In [ ]:
import os
from pathlib import Path

# ── Caminhos
PASTA_ENTRADA = Path('/content/drive/MyDrive/fotos_obra')
PASTA_SAIDA   = Path('/content/drive/MyDrive/fotos_obra_ferrugem')
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

# ── Parâmetros HSV (ajuste aqui para toda a pipeline)
HMIN = 5    # matiz mínima — laranja começa ~5 no OpenCV (0–179)
HMAX = 30   # matiz máxima — marrom-ferrugem termina ~30
SMIN = 80   # saturação mínima — filtra cinzas e brancos
VMIN = 40   # brilho mínimo — aceita ferrugem em sombra

# ── Redimensionamento (0 = sem resize)
LARGURA_MAX = 1200

# ── Extensões aceitas
EXTENSOES = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif')

# ── Verificar pasta
if not PASTA_ENTRADA.exists():
    print(f'⛔ Pasta não encontrada: {PASTA_ENTRADA}')
    print('   Verifique o caminho ou adicione atalho conforme instrução acima.')
else:
    arquivos = sorted([p for p in PASTA_ENTRADA.iterdir() if p.suffix.lower() in EXTENSOES])
    print(f'📂 Entrada  : {PASTA_ENTRADA}')
    print(f'📁 Saída    : {PASTA_SAIDA}')
    print(f'⚙️  H=[{HMIN},{HMAX}]  S≥{SMIN}  V≥{VMIN}')
    print(f'🖼️  {len(arquivos)} imagem(ns) encontrada(s)')

---
## Seção 3 — Processamento em lote

### Pipeline completo passo a passo

```
Imagem colorida (BGR)
        │
        ▼
  GaussianBlur (5×5)       ← suaviza ruído antes de converter cor
        │
        ▼
  Conversão BGR → HSV      ← separa cor (H) de brilho (V)
        │
        ▼
  CLAHE no canal V         ← corrige iluminação desigual (mesmo CLAHE do módulo anterior!)
        │
        ▼
  cv2.inRange(lower, upper)← máscara binária: 255 onde é ferrugem, 0 no resto
        │
        ▼
  MORPH_OPEN  (1 iter)     ← remove pontos isolados (ruído)
  MORPH_CLOSE (2 iter)     ← fecha buracos dentro das manchas
        │
        ▼
  findContours             ← delimita regiões corroídas
        │
        ├── máscara PNG
        ├── overlay PNG (mancha destacada em laranja)
        ├── grid 2×3 PNG  (original / HSV / cinza / máscara / overlay / contorno)
        └── métrica: % de área corroída
```

### Por que CLAHE no canal V?

Relembre o módulo anterior: CLAHE equaliza o contraste localmente. Aplicado ao canal V do HSV, ele **normaliza o brilho** sem tocar na informação de cor (H e S). Resultado: pixels de ferrugem em regiões escuras — que teriam V baixo e seriam filtrados por `vmin` — passam a ser detectados corretamente.

In [ ]:
from tqdm.notebook import tqdm

resultados = []  # armazena métricas para relatório final
erros = []

for arq in tqdm(arquivos, desc='Processando imagens'):
    try:
        img = cv2.imread(str(arq))
        assert img is not None, 'Não foi possível abrir'

        # Redimensionar se necessário
        if LARGURA_MAX > 0 and img.shape[1] > LARGURA_MAX:
            r   = LARGURA_MAX / img.shape[1]
            img = cv2.resize(img, (LARGURA_MAX, int(img.shape[0] * r)),
                             interpolation=cv2.INTER_AREA)

        mask, overlay, contoured, pct = detectar_ferrugem(
            img, HMIN, HMAX, SMIN, VMIN)

        base = arq.stem

        # Grid 2×3: original | HSV | cinza | máscara | overlay | contorno
        gray    = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        hsv_viz = cv2.cvtColor(cv2.cvtColor(img, cv2.COLOR_BGR2HSV), cv2.COLOR_HSV2BGR)
        top = np.hstack([img, hsv_viz, cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)])
        bot = np.hstack([cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR), overlay, contoured])
        grid = np.vstack([top, bot])

        # Texto de métrica no grid
        cv2.rectangle(grid, (10, 10), (500, 65), (255, 255, 255), -1)
        cv2.putText(grid, f'Area corroida: {pct:.2f}%', (20, 52),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 0), 2, cv2.LINE_AA)

        cv2.imwrite(str(PASTA_SAIDA / f'{base}_mask.png'), mask)
        cv2.imwrite(str(PASTA_SAIDA / f'{base}_overlay.png'), contoured)
        cv2.imwrite(str(PASTA_SAIDA / f'{base}_grid.png'), grid)

        resultados.append({'arquivo': arq.name, 'area_pct': pct})

    except Exception as e:
        erros.append((arq.name, str(e)))

print(f'\n✅ {len(resultados)} imagem(ns) processada(s) em: {PASTA_SAIDA}')
if erros:
    print('\n⚠️  Arquivos com erro:')
    for nome, msg in erros:
        print(f'   {nome}: {msg}')

---
## Seção 4 — Visualização e métricas

### 4a. Grid de resultados

Cada linha do grid mostra, para uma imagem:

| Coluna | Conteúdo |
|--------|----------|
| 1 | Original (BGR) |
| 2 | Espaço HSV (visualizado como BGR) — note como o H separa a ferrugem |
| 3 | Máscara binária — branco = pixels detectados como ferrugem |
| 4 | Overlay — mancha laranja sobre a imagem original |

**O que observar:** a máscara deve cobrir as manchas laranja-marrom e **não** detectar partes cinzas do concreto, armaduras limpas ou vegetação.

In [ ]:
MAX_EXIBIR = 4
amostra = arquivos[:MAX_EXIBIR]

for arq in amostra:
    img  = cv2.imread(str(arq))
    if img is None:
        continue
    if LARGURA_MAX > 0 and img.shape[1] > LARGURA_MAX:
        r   = LARGURA_MAX / img.shape[1]
        img = cv2.resize(img, (LARGURA_MAX, int(img.shape[0] * r)), interpolation=cv2.INTER_AREA)

    mask, overlay, contoured, pct = detectar_ferrugem(img, HMIN, HMAX, SMIN, VMIN)
    hsv_viz = cv2.cvtColor(cv2.cvtColor(img, cv2.COLOR_BGR2HSV), cv2.COLOR_HSV2BGR)

    fig, eixos = plt.subplots(1, 4, figsize=(18, 4))
    dados = [
        (img,      'Original',          None),
        (hsv_viz,  'Espaço HSV',         None),
        (mask,     'Máscara ferrugem',   'gray'),
        (contoured,f'Overlay — {pct:.1f}% corroído', None),
    ]
    for ax, (im, titulo, cmap) in zip(eixos, dados):
        ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB) if cmap is None else im, cmap=cmap)
        ax.set_title(titulo, fontsize=9)
        ax.axis('off')

    plt.suptitle(arq.name, fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.show()

### 4b. Relatório de métricas — ranking de imagens por área corroída

A tabela abaixo ordena as imagens da pasta do maior para o menor percentual de área corroída detectada. Use isso para **priorizar inspeções** — imagens com alto percentual merecem atenção imediata.

> ⚠️ **Interpretação cuidadosa:** um percentual alto pode indicar também **falso positivo** (ex: pintura laranja, argamassa colorida). Sempre cruze com a visualização da máscara.

In [ ]:
import pandas as pd

df = pd.DataFrame(resultados).sort_values('area_pct', ascending=False).reset_index(drop=True)
df.index += 1
df.columns = ['Arquivo', 'Área corroída (%)']
df['Área corroída (%)'] = df['Área corroída (%)'].round(2)

# Destaque visual por severidade
def destacar(val):
    if val > 20:  return 'background-color: #ffcccc'  # vermelho leve
    if val > 10:  return 'background-color: #ffe0b2'  # laranja leve
    if val > 3:   return 'background-color: #fff9c4'  # amarelo leve
    return ''

display(df.style.applymap(destacar, subset=['Área corroída (%)']))

media = df['Área corroída (%)'].mean()
print(f'\n📊 Média da pasta: {media:.2f}%')
print(f'   Máximo       : {df["Área corroída (%)"].max():.2f}%  ({df.iloc[0]["Arquivo"]})')

---
## Seção 5 — Calibração de parâmetros HSV

### Entendendo cada parâmetro

| Parâmetro | Efeito de aumentar | Efeito de diminuir |
|-----------|-------------------|-------------------|
| `hmax` | Inclui tons mais escuros (marrom) | Restringe a laranja puro |
| `smin` | Exclui cores "desbotadas" | Inclui paredes ocre, poeira |
| `vmin` | Exclui sombras | Detecta ferrugem em regiões escuras |

A grade abaixo varia `hmax` (linhas) e `smin` (colunas) para a primeira imagem da pasta.
Use o resultado para atualizar `HMAX` e `SMIN` na Seção 2 e reprocessar.

In [ ]:
if arquivos:
    img_ref = cv2.imread(str(arquivos[0]))
    if LARGURA_MAX > 0 and img_ref.shape[1] > LARGURA_MAX:
        r       = LARGURA_MAX / img_ref.shape[1]
        img_ref = cv2.resize(img_ref, (LARGURA_MAX, int(img_ref.shape[0] * r)),
                             interpolation=cv2.INTER_AREA)

    hmaxs = [20, 25, 30, 35]
    smins = [60, 80, 100, 120]

    n_lin = len(hmaxs)
    n_col = len(smins)

    fig, eixos = plt.subplots(n_lin, n_col, figsize=(4.5 * n_col, 4 * n_lin))

    for i, hm in enumerate(hmaxs):
        for j, sm in enumerate(smins):
            mask, _, contoured, pct = detectar_ferrugem(
                img_ref, hmin=HMIN, hmax=hm, smin=sm, vmin=VMIN)
            eixos[i][j].imshow(cv2.cvtColor(contoured, cv2.COLOR_BGR2RGB))
            eixos[i][j].set_title(f'hmax={hm}  smin={sm}\n{pct:.1f}% corroído', fontsize=8)
            eixos[i][j].axis('off')

    plt.suptitle(f'Grade de parâmetros HSV — {arquivos[0].name}',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print('👆 Escolha a combinação com melhor cobertura sem falsos positivos.')
    print('   Atualize HMAX e SMIN na Seção 2 e execute novamente a Seção 3.')

---
## Próximos passos

Com os resultados desta etapa, você tem:

- **Máscara binária por imagem** → entrada para modelos de deep learning (U-Net como dado de treino com pseudo-label)
- **Percentual de área corroída** → campo quantitativo para o BIM (Pset_SAC, atributo `grau_corrosao`)
- **Contornos delimitados** → base para geração de BCF Issues no Solibri

```
fotos_obra_ferrugem/
        │
        ├── Treino U-Net de segmentação de corrosão
        ├── Associação com elemento IFC (por localização na foto)
        └── Registro automático de não-conformidade → BCF Issue
```

### Referências

- Gonzalez, R. C.; Woods, R. E. (2018). *Digital Image Processing*, 4ª ed. — Cap. 6 (Color Image Processing)
- OpenCV docs: [`cv2.inRange`](https://docs.opencv.org/4.x/da/d97/tutorial_threshold_inRange.html), [`cv2.morphologyEx`](https://docs.opencv.org/4.x/d9/d61/tutorial_py_morphological_ops.html)
- Helene, P. R. L. (1993). *Manual para Reparo, Reforço e Proteção de Estruturas de Concreto* — Regra de Sitter / Lei dos 5
- ABNT NBR 6118:2014 — Projeto de estruturas de concreto: procedimento
